1. 字符编码： 汉字 -> unicode -> UTF-8

2. 词元化：tokenization

若按“完整词（Word）”切分

主要缺陷：词表爆炸与“未登录词”问题（Out-of-Vocabulary, OOV）

词表空间爆炸： 如果要涵盖所有合法的词汇（包括各种变体、复数、拼写错误、专业术语、跨语言词汇），词表规模会大到难以维护。

灾难性的 OOV： 这是最致命的。一旦模型遇到词表中没有的词（例如罕见的人名、新造词、或带有拼写错误的输入），模型就会将其视为“未知（UNK）”。这直接导致模型彻底“失明”，无法对该词进行任何推理，因为该词完全不在其概率分布的训练空间内。

4个special token

<PAD> (Padding Token) 模型要求一个 Batch内的文本长度必须一致。如果某句话太短，就在末尾补上 <PAD>

<UNK> (Unknown Token) 当输入文本中出现了词表（Vocabulary）里没有的词时，统一用 <UNK> 代替

<BOS> / <SOS> (Beginning of Sentence) 句子开始标记。

<EOS> (End of Sentence) 句子结束标记。

BPE (Byte Pair Encoding，字节对编码) 

- **逻辑**：一种自底向上（Bottom-up）的迭代合并算法，通过统计频率将最常出现的相邻字符对合并。
  
### 2. 算法三大步骤
1. **初始化**：将所有单词拆分为单个字符，并在末尾添加结束符 `</w>`（标记单词边界）。
2. **统计频率**：在语料库中统计所有相邻 Token 对（Pair）出现的总次数。
3. **迭代合并**：
   - 找出频率最高的对（如 'e' 和 's'）。
   - 将其合并为一个新 Token（'es'）加入词表。
   - 更新语料库，将所有的 'e s' 替换为 'es'。
4. **终止条件**：达到预设的词表大小（Vocab Size）或迭代次数。

---

### 3. BPE 流程演示 (可视化)
假设语料库统计：{"hug": 10, "pug": 5, "pun": 12, "bun": 4}

# 初始状态 (Tokens)
- h u g </w> (10)
- p u g </w> (5)
- p u n </w> (12)
- b u n </w> (4)
# 初始词表: [h, u, g, p, n, b, </w>]

# 第一次迭代:
- 统计发现 (u, n) 出现 12+4=16 次（最高）
- 合并 (u, n) -> "un"
- 词表更新: [..., un]

# 第二次迭代:
- 统计发现 (un, </w>) 出现 16 次（最高）
- 合并 (un, </w>) -> "un</w>"
- 词表更新: [..., un, un</w>]

# 最终效果:
- 常用词如 "the", "ing" 会合并为完整 Token。
- 罕见词如 "unhappily" 会被拆分为 ["un", "happi", "ly"]。

---

### 4. 关键优势
1. **解决 OOV (Out-of-Vocabulary) 问题**：
   - 即使遇到从未见过的单词，只要它的字符/子词在词表里，就能被组合出来。
2. **压缩序列长度**：
   - 相比字符级，序列更短，模型计算效率更高。
3. **捕捉形态学特征**：
   - 自动提取前缀（un-, re-）、后缀（-ing, -ed）和词根。
4. **Byte-level BPE (BBPE)**：
   - GPT-2/GPT-3 使用版本。从 256 个字节开始合并，词表更小，支持任何 UTF-8 字符（Emoji, 中文等）。

# NLP 笔记：BPE 的缺点与局限性

### 1. 缺乏形态学逻辑 (Linguistic Ignorance)
- **问题**：BPE 完全基于统计频率进行合并，不考虑语言的语法或语义结构。
- **表现**：它可能会把一个单词切分成对人类来说毫无意义的片段。
  - *例子*：单词 "bi-lingual"（双语）。
  - *理想切分*：["bi", "lingual"]。
  - *BPE 可能切分*：["bilin", "gual"]（仅仅因为 "bilin" 在语料库中刚好凑巧高频）。
- **后果**：模型可能难以通过子词有效地学习词根和词缀的真正含义。


### 1. 为什么要在词尾添加结束符 `</w>`？
- **防止跨词合并**：确保 BPE 的合并操作仅发生在“单词内部”，而不会把前一个词的词尾和后一个词的词头连在一起（例如：防止 `the cat` 中的 `e` 和 `c` 合并）。
- **保证还原一致性**：在解码阶段（Tokens -> Text），`</w>` 明确标记了单词的边界，方便将子词重新拼接回原始句子。
- **简化统计**：方便程序识别哪些片段是完整的词尾，哪些是词首或词中。

# WordPiece 算法

### 1. 基本定义
- **代表模型**：BERT, DistilBERT, Electra.
- **核心逻辑**：基于概率（Likelihood）而非单纯频率的子词算法。
- **标记风格**：使用 `##` 表示子词（非首位片段）。

---

### 2. 核心公式 (Merge Strategy)
WordPiece 选择合并 pair(A, B) 的依据是：
$Score = Count(AB) / (Count(A) * Count(B))$
- **直观理解**：分母越大，Score 越小。如果 A 和 B 本身是非常高频的常用字（如 'a', 'the'），它们被合并的优先级会降低，除非它们组成的 'ab' 极其频繁。

---

# WordPiece 编码逻辑总结

### 1. 编码策略：贪心匹配 (Greedy Matching)
- **原则**：Longest-Match-First (最长优先)。
- **操作**：从左往右扫描，每次都咬下“词表里能匹配到的最长一口”。

### 2. 状态约束 (The ## Convention)
- 一个单词被切分后：
  - **首个 Token**：必须是普通形式 (如 "low")，代表 Root。
  - **后续 Tokens**：必须带 "##" 前缀 (如 "##er")，代表 Suffix 或延续。
- **意义**：这种设计让模型一眼就能看出哪些 Token 属于同一个词。

### 3. 与 BPE 的区别 (面试常考)
- **BPE**：在分词时，按照 Merge List 的先后顺序，在全句范围内依次合并。
- **WordPiece**：在分词时，对每个单词独立进行最长匹配扫描。



Unigram 

- **分词概率**：P(Sentence) = P(t1) * P(t2) * ... * P(tn)
- **选择准则**：寻找使上述乘积最大的切分路径 (Argmax)。


- 输入词：`unhappy`
- 路径 1：`[unhappy]` -> P=0.001
- 路径 2：`[un, happy]` -> P(un)*P(happy) = 0.01 * 0.2 = 0.002
- 决策：选路径 2，因为 0.002 > 0.001。
